In [1]:
import itertools
import yaml
import torch
from pathlib import Path
from neuralhydrology.nh_run import start_run

In [ ]:
precip_products = [
    "prcp_mm_day",
    "prcp_chirps_mm_day",
    "prcp_mswep_mm_day",
    "prcp_gauge_mm_day",
]

# seeds = [111, 222, 333, 444]
seeds = [555, 666, 777, 888]

In [3]:
base_config_path = Path("training.yml")

with open(base_config_path, "r") as f:
    base_config = yaml.safe_load(f)

In [4]:
for r in range(1, len(precip_products) + 1):
    for precip_combo in itertools.combinations(precip_products, r):
        for seed in seeds:

            config = base_config.copy()

            # Update seed
            config["seed"] = seed

            # Update dynamic inputs:
            # Keep temperature and radiation, replace precip
            config["dynamic_inputs"] = [
                "srad_W_m2",
                "tmax_C",
                "tmin_C",
                *precip_combo
            ]

            # Create unique experiment name
            precip_name = "_".join(precip_combo)
            config["experiment_name"] = (
                f"precip_{precip_name}_seed_{seed}"
            )

            # Save temporary config file
            temp_config_path = Path(f"temp_{precip_name}_seed_{seed}.yml")
            with open(temp_config_path, "w") as f:
                yaml.dump(config, f)

            print(f"Running: {config['experiment_name']}")

            # Run training
            if torch.cuda.is_available() or torch.backends.mps.is_available():
                start_run(config_file=temp_config_path)
            else:
                start_run(config_file=temp_config_path, gpu=-1)

Running: precip_prcp_mm_day_seed_111
2026-02-22 18:45:33,358: Logging to /home/azureuser/sky_workdir/final_runs/runs/precip_prcp_mm_day_seed_111_2202_184533/output.log initialized.
2026-02-22 18:45:33,358: ### Folder structure created at /home/azureuser/sky_workdir/final_runs/runs/precip_prcp_mm_day_seed_111_2202_184533
2026-02-22 18:45:33,359: ### Run configurations for precip_prcp_mm_day_seed_111
2026-02-22 18:45:33,360: batch_size: 256
2026-02-22 18:45:33,360: clip_gradient_norm: 1
2026-02-22 18:45:33,361: data_dir: data
2026-02-22 18:45:33,361: dataset: generic
2026-02-22 18:45:33,362: device: cuda:0
2026-02-22 18:45:33,362: dynamic_inputs: ['srad_W_m2', 'tmax_C', 'tmin_C', 'prcp_mm_day']
2026-02-22 18:45:33,362: epochs: 30
2026-02-22 18:45:33,363: experiment_name: precip_prcp_mm_day_seed_111
2026-02-22 18:45:33,363: head: regression
2026-02-22 18:45:33,364: hidden_size: 256
2026-02-22 18:45:33,364: initial_forget_bias: 5
2026-02-22 18:45:33,365: learning_rate: {0: 0.001}
2026-02-2